# MovieLens 100k 추천 시스템 프로젝트

## 1. 데이터 로딩 및 전처리

In [1]:
import pandas as pd
import numpy as np

# u.data 로딩 (user_id, item_id, rating, timestamp)
df = pd.read_csv("ml-100k/u.data", sep='\t', names=["user_id", "item_id", "rating", "timestamp"])

# 훈련 및 테스트 분할
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# 사용자-아이템 평점 행렬 생성
pivot_table = train_df.pivot(index='user_id', columns='item_id', values='rating')
pivot_table.fillna(0, inplace=True)
R = pivot_table.values


## 2. 협업 필터링: User-Based, Item-Based

In [2]:
from sklearn.metrics.pairwise import cosine_similarity

# User-User 기반
user_sim = cosine_similarity(R)
user_pred = user_sim @ R / np.array([np.abs(user_sim).sum(axis=1)]).T

# Item-Item 기반
item_sim = cosine_similarity(R.T)
item_pred = R @ item_sim / np.array([np.abs(item_sim).sum(axis=1)])

print("User-Based CF 예측 행렬 shape:", user_pred.shape)
print("Item-Based CF 예측 행렬 shape:", item_pred.shape)


User-Based CF 예측 행렬 shape: (943, 1653)
Item-Based CF 예측 행렬 shape: (943, 1653)


## 3. 행렬 분해 모델: SVD

In [3]:
from scipy.sparse.linalg import svds

# 평점 평균 중심화
R_mean = R.mean(axis=1).reshape(-1, 1)
R_demeaned = R - R_mean

# SVD 분해
U, sigma, Vt = svds(R_demeaned, k=20)
sigma = np.diag(sigma)

# 예측 복원
svd_pred = U @ sigma @ Vt + R_mean


## 4. ALS 모델 (implicit library 사용)

In [6]:
# ALS는 implicit 패키지를 활용해 구현 가능
# pip install implicit 필요
from scipy.sparse import csr_matrix
import implicit

# 행렬 변환
R_sparse = csr_matrix(R)

# 모델 학습 (Implicit ALS는 사용자-아이템 아닌 아이템-사용자 형태)
als_model = implicit.als.AlternatingLeastSquares(factors=20, iterations=15)
als_model.fit(R_sparse.T)

# 예측 점수
user_id = 0  # 예시 사용자 1번 (index 0)
recommended = als_model.recommend(user_id, R_sparse[user_id], N=5)
print("ALS 추천 예시 (user 1):", recommended)


C:\Users\shjun\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\shjun\AppData\Roaming\Python\Python310\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
C:\Users\shjun\AppData\Roaming\Python\Python310\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.0013892650604248047 seconds
  warnings.warn(
100%|██████████| 15/15 [00:00<00:00, 74.61it/s]

ALS 추천 예시 (user 1): (array([886,  81, 647, 343, 653]), array([1.3382733, 1.2435473, 1.2378988, 1.1756582, 1.1648297],
      dtype=float32))


## 5. Autoencoder 기반 추천 시스템

In [7]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

num_users, num_items = R.shape

input_layer = Input(shape=(num_items,))
encoded = Dense(128, activation='relu')(input_layer)
encoded = Dense(64, activation='relu')(encoded)
decoded = Dense(num_items, activation='linear')(encoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.fit(R, R, epochs=20, batch_size=64, verbose=0)

autoencoder_pred = autoencoder.predict(R)


30/30 [==============================] - 0s 2ms/step


## 6. 모델 평가: RMSE

In [8]:
from sklearn.metrics import mean_squared_error

def rmse(true_df, pred_matrix):
    preds = []
    trues = []
    for row in true_df.itertuples():
        u, i, r = row.user_id, row.item_id, row.rating
        try:
            pred = pred_matrix[u-1, i-1]
            preds.append(pred)
            trues.append(r)
        except:
            continue
    return np.sqrt(mean_squared_error(trues, preds))

print("RMSE - User-Based CF:", rmse(test_df, user_pred))
print("RMSE - Item-Based CF:", rmse(test_df, item_pred))
print("RMSE - SVD:", rmse(test_df, svd_pred))
print("RMSE - Autoencoder:", rmse(test_df, autoencoder_pred))


RMSE - User-Based CF: 3.005964201918961
RMSE - Item-Based CF: 3.1326552710723017
RMSE - SVD: 2.6784725374469507
RMSE - Autoencoder: 2.737848632850288


## 7. 사용자 만족도 평가 (Top-N Precision / Recall)

In [9]:
# 사용자 만족도 평가는 추천 Top-N 리스트와 실제 평점 데이터를 비교
# 간단한 Precision@5 구현 예시

def precision_at_k(user_id, pred_matrix, true_df, k=5):
    item_scores = list(enumerate(pred_matrix[user_id-1]))
    top_items = sorted(item_scores, key=lambda x: x[1], reverse=True)[:k]
    top_item_ids = [i+1 for i, _ in top_items]
    relevant_items = true_df[(true_df.user_id == user_id) & (true_df.rating >= 4)].item_id.tolist()
    hit = len(set(top_item_ids) & set(relevant_items))
    return hit / k

print("Precision@5 (User 1) - SVD:", precision_at_k(1, svd_pred, test_df, k=5))


Precision@5 (User 1) - SVD: 0.0


### 🔍 Precision@5이 0.0이란 의미는?
#### SVD 모델이 추천한 상위 5개 아이템 중에

#### 실제 사용자(User 1)가 4점 이상을 준 영화가

#### 하나도 포함되지 않았다는 뜻이에요.

### ✅ 가능한 원인
#### 1. User 1의 평가 수가 적거나 없음
#### test_df에 있는 User 1의 실제 평점 중 4점 이상이 거의 없거나 아예 없을 수도 있음.

#### 2. SVD 모델 예측값이 낮게 나왔거나 추천이 무의미한 항목 포함
#### 평점 평균 중심화 방식이나 train_test_split 후 학습 데이터에 해당 아이템 정보가 부족했을 수 있어.

#### 3. test_df 기준의 평가가 너무 희박함 (sparse)
#### 특히 Top-N 평가는 추천 결과에 따라 영향을 많이 받음.